In [0]:
%sql
SELECT
    warehouse_id AS armazem,
    COUNT(*) AS total_pedidos,
    SUM(CASE
        WHEN late_delivery = 'Yes' THEN 1
        ELSE 0
    END) AS pedidos_atrasados,
    ROUND(
        SUM(CASE
            WHEN late_delivery = 'Yes' THEN 1
            ELSE 0
        END) * 100.0 / COUNT(*),
        2
    ) AS percentual_atraso
FROM workspace.default.entregas_tratadas
GROUP BY warehouse_id
ORDER BY percentual_atraso DESC;

Os percentuais de atraso variaram de 54,32% no WH-010 a 57,59% no WH-008, uma diferença de 3,27 pontos percentuais. Embora o WH-008 apresente o maior percentual, todos os armazéns possuem taxas elevadas e relativamente próximas. É necessário considerar características dos pedidos, como distância e prazo prometido, antes de atribuir as diferenças exclusivamente ao desempenho operacional.

In [0]:
%sql
SELECT
    carrier AS transportadora,
    COUNT(*) AS total_pedidos,
    SUM(CASE
        WHEN late_delivery = 'Yes' THEN 1
        ELSE 0
    END) AS pedidos_atrasados,
    ROUND(
        SUM(CASE
            WHEN late_delivery = 'Yes' THEN 1
            ELSE 0
        END) * 100.0 / COUNT(*),
        2
    ) AS percentual_atraso,
    ROUND(
        AVG(CASE
            WHEN late_delivery = 'Yes'
            THEN delivery_delay_days
        END),
        2
    ) AS media_dias_atraso
FROM workspace.default.entregas_tratadas
GROUP BY carrier
ORDER BY percentual_atraso DESC;


A GlobalExpress apresentou o maior percentual de atraso (65,75%) e a maior média de dias de atraso (2,46). Em contrapartida, a SwiftShip registrou o menor percentual (47,34%) e a menor média (2,08 dias). Os resultados indicam diferenças de desempenho entre as transportadoras, principalmente na frequência dos atrasos. Entretanto, antes de recomendar mudanças de parceiros logísticos, é necessário verificar se essas diferenças permanecem quando consideramos fatores como distância percorrida e prazo prometido.


In [0]:
%sql
-- distancia X atrasos
SELECT
    carrier AS transportadora,
    CASE
        WHEN distance_km < 500 THEN 'Até 499 km'
        WHEN distance_km < 1000 THEN '500 a 999 km'
        WHEN distance_km < 1500 THEN '1000 a 1499 km'
        ELSE '1500 km ou mais'
    END AS faixa_distancia,
    COUNT(*) AS total_pedidos,
    ROUND(
        SUM(CASE
            WHEN late_delivery = 'Yes' THEN 1
            ELSE 0
        END) * 100.0 / COUNT(*),
        2
    ) AS percentual_atraso
FROM workspace.default.entregas_tratadas
GROUP BY
    carrier,
    CASE
        WHEN distance_km < 500 THEN 'Até 499 km'
        WHEN distance_km < 1000 THEN '500 a 999 km'
        WHEN distance_km < 1500 THEN '1000 a 1499 km'
        ELSE '1500 km ou mais'
    END
ORDER BY faixa_distancia, percentual_atraso DESC;

Ao comparar as transportadoras dentro de faixas de distância semelhantes, as diferenças de desempenho permaneceram. A GlobalExpress apresentou os maiores percentuais de atraso em todas as faixas, enquanto a SwiftShip apresentou os menores. Isso indica que a distância, isoladamente, não explica as diferenças observadas entre as transportadoras. Entretanto, ainda precisamos considerar outras características, especialmente o prazo prometido, antes de concluir que uma transportadora apresenta desempenho operacional inferior.

In [0]:
%sql
-- prazo prometido X atrasos

SELECT
    carrier AS transportadora,
    promised_delivery_days AS prazo_prometido_dias,
    COUNT(*) AS total_pedidos,
    ROUND(
        SUM(CASE
            WHEN late_delivery = 'Yes' THEN 1
            ELSE 0
        END) * 100.0 / COUNT(*),
        2
    ) AS percentual_atraso
FROM workspace.default.entregas_tratadas
GROUP BY
    carrier,
    promised_delivery_days
ORDER BY
    promised_delivery_days,
    percentual_atraso DESC;

In [0]:
%sql
-- prazo prometido X atrasos

SELECT
    carrier AS transportadora,
    CASE
        WHEN promised_delivery_days <= 3 THEN '1. Até 3 dias'
        WHEN promised_delivery_days <= 6 THEN '2. 4 a 6 dias'
        WHEN promised_delivery_days <= 10 THEN '3. 7 a 10 dias'
        ELSE '4. 11 dias ou mais'
    END AS faixa_prazo,
    COUNT(*) AS total_pedidos,
    SUM(
        CASE
            WHEN late_delivery = 'Yes' THEN 1
            ELSE 0
        END
    ) AS pedidos_atrasados,
    ROUND(
        SUM(
            CASE
                WHEN late_delivery = 'Yes' THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS percentual_atraso
FROM workspace.default.entregas_tratadas
GROUP BY
    carrier,
    CASE
        WHEN promised_delivery_days <= 3 THEN '1. Até 3 dias'
        WHEN promised_delivery_days <= 6 THEN '2. 4 a 6 dias'
        WHEN promised_delivery_days <= 10 THEN '3. 7 a 10 dias'
        ELSE '4. 11 dias ou mais'
    END
ORDER BY
    faixa_prazo,
    percentual_atraso DESC;


-  A GlobalExpress apresenta a maior taxa de atraso nas quatro faixas.
Isso mostra que seu percentual de atraso permanece mais alto mesmo quando comparamos pedidos com prazos prometidos semelhantes. Portanto, a diferença observada na análise geral não parece ser explicada apenas pelo prazo prometido.
-  A SwiftShip apresenta a menor taxa de atraso nas quatro faixas.
Ela mantém o menor percentual de atraso em todos os grupos, o que reforça o resultado da análise anterior.
-  A taxa de atraso é maior nas faixas de prazo mais longas.
Isso não significa necessariamente que prazos maiores provoquem atrasos.

In [0]:
%sql
--cruzar transportadora, faixa de distância e faixa de prazo prometido.
SELECT
    carrier AS transportadora,

    CASE
        WHEN distance_km < 500 THEN '1. Até 499 km'
        WHEN distance_km < 1000 THEN '2. 500 a 999 km'
        WHEN distance_km < 1500 THEN '3. 1000 a 1499 km'
        ELSE '4. 1500 km ou mais'
    END AS faixa_distancia,

    CASE
        WHEN promised_delivery_days <= 3 THEN '1. Até 3 dias'
        WHEN promised_delivery_days <= 6 THEN '2. 4 a 6 dias'
        WHEN promised_delivery_days <= 10 THEN '3. 7 a 10 dias'
        ELSE '4. 11 dias ou mais'
    END AS faixa_prazo,

    COUNT(*) AS total_pedidos,

    SUM(
        CASE
            WHEN late_delivery = 'Yes' THEN 1
            ELSE 0
        END
    ) AS pedidos_atrasados,

    ROUND(
        SUM(
            CASE
                WHEN late_delivery = 'Yes' THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS percentual_atraso

FROM workspace.default.entregas_tratadas

GROUP BY
    carrier,
    CASE
        WHEN distance_km < 500 THEN '1. Até 499 km'
        WHEN distance_km < 1000 THEN '2. 500 a 999 km'
        WHEN distance_km < 1500 THEN '3. 1000 a 1499 km'
        ELSE '4. 1500 km ou mais'
    END,
    CASE
        WHEN promised_delivery_days <= 3 THEN '1. Até 3 dias'
        WHEN promised_delivery_days <= 6 THEN '2. 4 a 6 dias'
        WHEN promised_delivery_days <= 10 THEN '3. 7 a 10 dias'
        ELSE '4. 11 dias ou mais'
    END

ORDER BY
    faixa_distancia,
    faixa_prazo,
    percentual_atraso DESC;


Mesmo ao comparar pedidos por faixas de distância e prazo prometido, foram observadas diferenças nas taxas de atraso entre as transportadoras. A GlobalExpress apresentou percentuais elevados em diversas combinações, enquanto a SwiftShip apresentou percentuais menores em muitas delas. Esses resultados indicam que vale investigar diferenças operacionais entre as transportadoras, sem atribuir causalidade apenas a essa análise.

In [0]:
%sql
-- cruzar armazen e transportadora


SELECT
    warehouse_id AS armazem,
    carrier AS transportadora,
    COUNT(*) AS total_pedidos,
    SUM(
        CASE
            WHEN late_delivery = 'Yes' THEN 1
            ELSE 0
        END
    ) AS pedidos_atrasados,
    ROUND(
        SUM(
            CASE
                WHEN late_delivery = 'Yes' THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS percentual_atraso
FROM workspace.default.entregas_tratadas
GROUP BY
    warehouse_id,
    carrier
ORDER BY
    pedidos_atrasados DESC;
